# CDS4003 – Data Science Final Project
## Notebook 04 – Stage 7: Deployment

**Prerequisite:** notebooks 01–03 completed (model, metadata and shared feature code exist in your Drive folder).
**Outputs:** `app/app.py`, `requirements.txt`, `README.md`, `.gitignore`, a tested app, and a final package for GitHub.

### Setup
The next cells load the libraries, connect to your Drive project folder and load the saved model and its metadata.

In [1]:
# ---- Libraries ----
import re
import sys
import json
import shutil
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.simplefilter("ignore", category=FutureWarning)      # hides harmless deprecation notices only
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 60)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

IN_COLAB = "google.colab" in sys.modules

# ---- Project folder in Google Drive (shared by notebooks 01-04) ----
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Used-Vehicle-Price-Prediction")
else:
    PROJECT_DIR = Path("Used-Vehicle-Price-Prediction")

RAW_DIR, PROCESSED_DIR, SAMPLE_DIR = PROJECT_DIR / "data/raw", PROJECT_DIR / "data/processed", PROJECT_DIR / "data/sample"
OUTPUT_DIR, FIG_DIR = PROJECT_DIR / "outputs", PROJECT_DIR / "outputs/figures"
NOTEBOOK_DIR, MODEL_DIR, APP_DIR, SRC_DIR, DOCS_DIR = (PROJECT_DIR / n for n in ["notebooks", "models", "app", "src", "docs"])

settings_path = OUTPUT_DIR / "cleaning_settings.json"
if not settings_path.exists():
    raise FileNotFoundError(f"{settings_path} not found. Run notebook 01 completely first (it saves the cleaned data and settings "
                            f"in {PROJECT_DIR}).")
for folder in [FIG_DIR, MODEL_DIR, APP_DIR, SRC_DIR, DOCS_DIR, NOTEBOOK_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ---- Settings decided in notebook 01 (the same rules are used everywhere) ----
cleaning_settings = json.loads(settings_path.read_text(encoding="utf-8"))
DATA_COLLECTION_YEAR = cleaning_settings["DATA_COLLECTION_YEAR"]
PRICE_MAX_PLAUSIBLE = cleaning_settings["PRICE_MAX_PLAUSIBLE"]
PRICE_LOW_REVIEW = cleaning_settings["PRICE_LOW_REVIEW"]

def std_name(name):
    """Standard column name: stripped, lowercase, spaces replaced by underscores."""
    return re.sub(r"\s+", "_", str(name).strip().lower())

def load_cleaned(usecols=None):
    """Read the cleaned dataset saved by notebook 01."""
    path = PROCESSED_DIR / "vehicles_cleaned.csv"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run notebook 01 completely first.")
    df = pd.read_csv(path, usecols=usecols, low_memory=False, keep_default_na=False, na_values=[""])
    if "posting_date" in df.columns:
        df["posting_date"] = pd.to_datetime(df["posting_date"], errors="coerce", utc=True)
    return df

def save_fig(name):
    """Save the current figure into outputs/figures (used as report evidence)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    path = FIG_DIR / f"{name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    return path

print(f"Project folder          : {PROJECT_DIR}")
print(f"Data-collection year    : {DATA_COLLECTION_YEAR}")
print(f"Records after cleaning  : {cleaning_settings['records_cleaned']:,} (from {cleaning_settings['records_raw']:,} raw records)")

Mounted at /content/drive
Project folder          : /content/drive/MyDrive/Used-Vehicle-Price-Prediction
Data-collection year    : 2026
Records after cleaning  : 240,186 (from 295,451 raw records)


In [2]:
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
try:
    import vehicle_features as vf
except ImportError as error:
    raise ImportError("src/vehicle_features.py not found. Run notebook 02 first.") from error

import joblib
model_path = MODEL_DIR / "price_model.joblib"
metadata_path = MODEL_DIR / "model_metadata.json"
if not model_path.exists() or not metadata_path.exists():
    raise FileNotFoundError("The saved model was not found. Run notebook 03 completely first (it saves the model in models/).")

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
loaded_model = joblib.load(model_path)
FINAL_MODEL_NAME = metadata["model_name"]
final_metrics = metadata["test_metrics"]
RATIO_LOW, RATIO_HIGH = metadata["ratio_low"], metadata["ratio_high"]

print(f"Final model          : {FINAL_MODEL_NAME}")
print(f"Model file size      : {model_path.stat().st_size / 1024**2:,.1f} MB")
print(f"Test R2 / MdAPE / within 20%: {final_metrics['R2']:.3f} / {final_metrics['MdAPE (%)']:.1f}% / {final_metrics['Within 20% (%)']:.1f}%")

Final model          : Hist. gradient boosting
Model file size      : 2.9 MB
Test R2 / MdAPE / within 20%: 0.676 / 18.7% / 52.7%


---
# STAGE 7 – Deployment

**Goal:** turn the final model into a public web application. Colab can **prepare and test everything**; the last step (publishing on Streamlit Community Cloud) is done on the Streamlit website, connected to your GitHub repository.

| Step | Where it happens | Section |
|---|---|---|
| Generate `app/app.py`, `requirements.txt`, `README.md`, `.gitignore` | Colab | 31 |
| Test the prediction logic and the app (automatic test, optional live preview) | Colab | 32 |
| Push the repository to GitHub (optional), publish on Streamlit Community Cloud, final package | Colab + GitHub + Streamlit website | 33 |

## 31. Generate the Streamlit App and Repository Files

**What will happen?** The following files are written into the project folder:

| File | Purpose |
|---|---|
| `app/app.py` | The Streamlit web application (inputs, validation, estimate, price range, asking-price check, model information) |
| `src/vehicle_features.py` | Already created in Stage 4 – the **same** feature code is used by the app, which prevents training/serving mismatch |
| `requirements.txt` | Libraries for Streamlit Community Cloud (versions pinned to this Colab session) |
| `README.md` | Documentation required by the rubric (fill in the placeholders: team members, links) |
| `.gitignore` | Keeps raw/large data and temporary files out of GitHub |

**App features that support the rubric:** validated inputs, a price estimate with a range, a plausibility warning, an optional *asking-price check* that shows the business value, a model-information tab with metrics and feature importance, and a clear explanation of limitations.

In [3]:
APP_CODE = r"""
# Used Vehicle Price Estimator - Streamlit application (CDS4003 Data Science Final Project)
import json
import sys
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT / "src"))
import vehicle_features as vf          # the same feature code that was used during training

st.set_page_config(page_title="Used Vehicle Price Estimator", layout="wide")


@st.cache_resource
def load_artifacts():
    price_model = joblib.load(ROOT / "models" / "price_model.joblib")
    meta = json.loads((ROOT / "models" / "model_metadata.json").read_text(encoding="utf-8"))
    return price_model, meta


try:
    price_model, meta = load_artifacts()
except Exception as error:
    st.error(f"The model could not be loaded: {error}")
    st.stop()

cats = meta["categories"]


def options(name):
    return cats.get(name) or ["unknown"]


def choose(label, name):
    # Show a menu only when the model knows more than one category (a field that is empty in the dataset is hidden).
    opts = options(name)
    return opts[0] if len(opts) == 1 else st.selectbox(label, opts)


st.title("Used Vehicle Price Estimator")
st.write("Estimate the typical **asking price** of a used car or truck in the US market, with an indicative price range, "
         "and check whether a seller's asking price looks fair.")

tab_estimate, tab_model, tab_business = st.tabs(["Estimate a price", "About the model", "Business use"])

with tab_estimate:
    left, right = st.columns(2)
    with left:
        manufacturer = st.selectbox("Manufacturer", options("manufacturer"))
        model_options = meta["models_by_manufacturer"].get(manufacturer, []) + ["unknown"]
        model_name = st.selectbox("Model", model_options)
        year = st.number_input("Model year", min_value=1950, max_value=vf.REFERENCE_YEAR + 1,
                               value=max(1950, vf.REFERENCE_YEAR - 6), step=1)
        odometer = st.number_input("Odometer (miles)", min_value=0, max_value=1_000_000, value=70_000, step=1_000)
        condition = choose("Condition", "condition")
        title_status = choose("Title status", "title_status")
        state = choose("State", "state")
    with right:
        vehicle_type = choose("Vehicle type", "type")
        fuel = choose("Fuel", "fuel")
        transmission = choose("Transmission", "transmission")
        drive = choose("Drive", "drive")
        cylinders = choose("Cylinders", "cylinders")
        size = choose("Size", "size")
        paint_color = choose("Paint colour", "paint_color")
        seller_type = choose("Seller type", "seller_type")

    asking_price = st.number_input("Asking price to check (optional, US dollars)", min_value=0, value=0, step=500)

    if st.button("Estimate price", type="primary"):
        values = {"year": year, "odometer": odometer, "manufacturer": manufacturer, "model": model_name,
                  "condition": condition, "cylinders": cylinders, "fuel": fuel, "title_status": title_status,
                  "transmission": transmission, "drive": drive, "size": size, "type": vehicle_type,
                  "paint_color": paint_color, "seller_type": seller_type, "state": state}
        problems = vf.validate_inputs(values)
        if problems:
            for message in problems:
                st.error(message)
        else:
            result = vf.predict_price(price_model, values, meta["ratio_low"], meta["ratio_high"])
            c1, c2, c3 = st.columns(3)
            c1.metric("Estimated asking price", f"${result['estimate']:,.0f}")
            c2.metric("Likely range (low)", f"${result['low']:,.0f}")
            c3.metric("Likely range (high)", f"${result['high']:,.0f}")
            st.caption("The range covers about 80% of historical asking prices for similar estimates.")

            age = max(vf.REFERENCE_YEAR - year, 1)
            if odometer / age > 40_000:
                st.warning("This mileage is unusually high for the vehicle's age; the estimate may be less reliable.")
            if manufacturer == "unknown" or model_name == "unknown":
                st.warning("Manufacturer or model is unknown, so the estimate is less specific.")

            if asking_price > 0:
                difference = (asking_price - result["estimate"]) / result["estimate"] * 100
                if asking_price < result["low"]:
                    st.success(f"The asking price is {abs(difference):.0f}% below the estimate - possibly a bargain, "
                               "or the vehicle has a problem that the inputs do not capture.")
                elif asking_price > result["high"]:
                    st.error(f"The asking price is {difference:.0f}% above the estimate and outside the likely range - "
                             "consider negotiating or comparing other listings.")
                else:
                    st.info(f"The asking price is within the likely range ({difference:+.0f}% from the estimate).")

with tab_model:
    st.subheader("How the estimate is made")
    st.write(f"A **{meta['model_name']}** model was trained on {meta['training_rows']:,} cleaned Craigslist listings "
             f"(data collected around {meta['reference_year']}). It predicts the logarithm of the price, so errors are "
             "judged in percentage terms.")
    metrics = meta["test_metrics"]
    table = pd.DataFrame({"Metric": ["R-squared", "Median absolute percentage error (%)", "Predictions within 20% (%)",
                                     "Average absolute error ($)"],
                          "Test-set value": [metrics["R2"], metrics["MdAPE (%)"], metrics["Within 20% (%)"], metrics["MAE ($)"]]})
    st.dataframe(table, hide_index=True)
    image_path = ROOT / "docs" / "feature_importance.png"
    if image_path.exists():
        st.image(str(image_path), caption="Which inputs the model relies on most (permutation importance)")
    st.subheader("Limitations")
    st.write("- The data contains **asking prices**, not final sale prices.\n"
             "- Listings come from one website and one time period; market prices change over time.\n"
             "- Seller-entered data can contain errors; rare and luxury vehicles are estimated less accurately.\n"
             "- The estimate does not know about accidents, modifications or the exact vehicle condition.")

with tab_business:
    st.subheader("How the tool supports decisions")
    st.write("- **Sellers:** choose a competitive asking price faster and avoid pricing far outside the market.\n"
             "- **Buyers:** check whether an asking price is fair before negotiating.\n"
             "- **Dealerships and marketplaces:** price inventory consistently and flag listings priced far from the typical range for review.")
    st.write("Use the estimate as a **reference point**, not as a guaranteed selling price, and re-train the model regularly "
             "with fresh listings because used-car prices drift.")
"""

APP_DIR.mkdir(parents=True, exist_ok=True)
(APP_DIR / "app.py").write_text(APP_CODE.lstrip("\n"), encoding="utf-8")
compile(APP_CODE, "app.py", "exec")                        # syntax check
print("Wrote app/app.py (syntax check passed).")

Wrote app/app.py (syntax check passed).


In [4]:
# requirements.txt - versions pinned to the libraries that TRAINED the model (saved in the metadata)
lv = metadata["library_versions"]
requirements_text = f"""streamlit>=1.35
pandas=={lv['pandas']}
numpy=={lv['numpy']}
scikit-learn=={lv['scikit-learn']}
joblib=={lv['joblib']}
scipy
"""
(PROJECT_DIR / "requirements.txt").write_text(requirements_text, encoding="utf-8")

gitignore_text = """# Large / raw data (GitHub rejects files above 100 MB)
data/raw/
data/processed/vehicles_cleaned.csv
# Python and notebook files
__pycache__/
*.pyc
.ipynb_checkpoints/
# Secrets - never commit these
.env
*.key
"""
(PROJECT_DIR / ".gitignore").write_text(gitignore_text, encoding="utf-8")

m = final_metrics
readme_text = f"""# Used Vehicle Price Prediction and Market Valuation Using Machine Learning

CDS4003 - Data Science Final Project

## Business problem
Buyers, sellers and dealerships find it hard to judge whether the asking price of a used vehicle is reasonable. This project builds a
regression model that estimates the typical asking price of a used car or truck from its attributes and deploys it as a web app.

## Team members and roles
| Name | GitHub username | Role / contribution |
|---|---|---|
| *add name* | *add username* | *add role* |
| *add name* | *add username* | *add role* |

## Dataset
* **Source:** Craigslist Cars and Trucks Dataset (Austin Reese) - https://www.kaggle.com/datasets/austinreese/craigslist-carstrucks-data
* **Licence:** *quote the licence shown on the Kaggle dataset page*
* **How to obtain:** download `vehicles.csv` from Kaggle and place it in `data/raw/` (it is not stored in this repository because of its size).
  A 5,000-row sample of the cleaned data is in `data/sample/`.

## Folder structure
```
data/        raw/ (not tracked), processed/ (full cleaned file not tracked), sample/
notebooks/   Colab notebook with Stages 1-7
src/         vehicle_features.py (feature engineering shared by training and app)
models/      price_model.joblib, model_metadata.json
app/         app.py (Streamlit app)
outputs/     CSV reports and figures
docs/        data dictionary, feature importance figure
requirements.txt
```

## How to run
1. Install the libraries: `pip install -r requirements.txt`
2. Run the notebook in `notebooks/` from top to bottom in Google Colab (upload `vehicles.csv` when asked).
3. Start the app locally: `streamlit run app/app.py`

## Summary of results
* Cleaned dataset: {cleaning_settings['records_cleaned']:,} listings after removing {cleaning_settings['duplicates_removed']:,} exact duplicates, {cleaning_settings.get('repeated_adverts_removed', 0):,} re-posted adverts and {cleaning_settings['price_records_excluded']:,} records with an invalid price.
* Final model: **{FINAL_MODEL_NAME}**, trained on log(price).
* Test set: R-squared {m['R2']:.3f}; median absolute percentage error {m['MdAPE (%)']:.1f}%; {m['Within 20% (%)']:.1f}% of predictions within 20% of the asking price;
  mean absolute error ${m['MAE ($)']:,.0f}.
* Limitations: asking prices (not sale prices), one website, one time period, seller-entered data.

## Live app
*Add the Streamlit Community Cloud link here after deployment.*

## AI tools used
*List the AI tools used in this project and how they were used (required by the course).*
"""
(PROJECT_DIR / "README.md").write_text(readme_text, encoding="utf-8")
print("Wrote requirements.txt, .gitignore and README.md")
print("\nrequirements.txt:\n" + requirements_text)

Wrote requirements.txt, .gitignore and README.md

requirements.txt:
streamlit>=1.35
pandas==2.2.3
numpy==2.4.6
scikit-learn==1.6.1
joblib==1.6.0
scipy



> **Python-version note:** Streamlit Community Cloud uses its own Python version. If installation fails because a pinned version does not exist there, loosen the pins in `requirements.txt` (for example `scikit-learn>=1.4`) – but the model must be loaded with the **same major scikit-learn version** it was saved with, so keep `scikit-learn` as close to the Colab version as possible.

## 32. Test the Prediction Logic and the App

**What will happen?**
1. The prediction function used by the app is tested on real test listings and on invalid inputs.
2. An **automatic app test** (Streamlit's `AppTest`) runs `app.py` without a browser, clicks the button and checks that no error occurs.
3. *(Optional)* a temporary live preview is opened through a tunnel.

**Why?** A deployed app that crashes on the first click earns very little credit. Testing before deployment finds errors early.

In [5]:
loaded_model = joblib.load(model_path)

# ---- Test 1: real listings ----
# Smoke test on rows of the saved 5,000-row sample (these rows may also have been used for training, so this checks the code, not accuracy)
sample_df = pd.read_csv(SAMPLE_DIR / "vehicles_cleaned_sample.csv", keep_default_na=False, na_values=[""])
usable = sample_df.dropna(subset=["year", "odometer"]).head(5).reindex(columns=vf.RAW_INPUT_COLUMNS)
rows = []
for idx, row in usable.iterrows():
    values = row.to_dict()
    result = vf.predict_price(loaded_model, values, RATIO_LOW, RATIO_HIGH)
    rows.append({"manufacturer": values["manufacturer"], "model": values["model"], "year": int(values["year"]),
                 "odometer": int(values["odometer"]), "listed asking price ($)": sample_df.loc[idx, "price"],
                 "estimate ($)": round(result["estimate"]), "range low ($)": round(result["low"]), "range high ($)": round(result["high"])})
display(pd.DataFrame(rows))

# ---- Test 2: input validation ----
bad_inputs = [{"year": 1800, "odometer": 50_000, "manufacturer": "ford"},
              {"year": 2015, "odometer": -10, "manufacturer": "ford"},
              {"year": 2015, "odometer": 50_000, "manufacturer": ""}]
for bad in bad_inputs:
    print(bad, "->", vf.validate_inputs(bad))
assert all(vf.validate_inputs(b) for b in bad_inputs), "Validation should reject every invalid input."

# ---- Test 3: unseen categories must not crash the model ----
odd = usable.iloc[0].to_dict()
odd.update({"manufacturer": "made-up-brand", "model": "made-up-model", "state": "zz"})
print("Prediction with unseen categories:", round(vf.predict_price(loaded_model, odd)["estimate"]))
print("All prediction-logic tests passed.")

,manufacturer,model,year,odometer,listed asking price ($),estimate ($),range low ($),range high ($)
0,ford,explorer xlt 4x4 suv,2016,151000,2200,4849,3205,8305
1,tesla,model s plaid,2021,63782,49114,33416,22082,57226
2,ram,promaster 3500 159 wb high roof extended,2017,215145,10995,9389,6204,16078
3,bmw,x3 28d xdrive,2015,92070,14990,12160,8036,20825
4,rover,unknown,1978,35000,15000,5526,3652,9464


{'year': 1800, 'odometer': 50000, 'manufacturer': 'ford'} -> ['Model year must be between 1950 and 2027.']
{'year': 2015, 'odometer': -10, 'manufacturer': 'ford'} -> ['Odometer must be between 0 and 1,000,000 miles.']
{'year': 2015, 'odometer': 50000, 'manufacturer': ''} -> ['Please choose a manufacturer.']
Prediction with unseen categories: 7432
All prediction-logic tests passed.


In [6]:
# Automatic app test (installs Streamlit if needed; skipped gracefully if that is not possible)
RUN_STREAMLIT_APP_TEST = True

if RUN_STREAMLIT_APP_TEST:
    try:
        try:
            import streamlit
        except ImportError:
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=True)
        from streamlit.testing.v1 import AppTest

        app_test = AppTest.from_file(str(APP_DIR / "app.py"), default_timeout=90)
        app_test.run()
        assert not app_test.exception, f"App failed on start: {app_test.exception}"
        app_test.button[0].click().run()
        assert not app_test.exception, f"App failed after clicking 'Estimate price': {app_test.exception}"
        print("App test passed: the app starts, and a price is estimated without errors.")
        print("Metrics shown by the app:", [(m.label, m.value) for m in app_test.metric])
    except Exception as error:
        print(f"App test could not be completed ({type(error).__name__}: {error}).")
        print("Run the app manually instead: streamlit run app/app.py")

2026-11-23 10:49:18.970 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


App test passed: the app starts, and a price is estimated without errors.
Metrics shown by the app: [('Estimated asking price', '$11,558'), ('Likely range (low)', '$7,638'), ('Likely range (high)', '$19,794')]


In [7]:
# OPTIONAL temporary live preview in Colab (needs internet; the link works only while this cell's session is running)
RUN_LIVE_PREVIEW = False

if RUN_LIVE_PREVIEW:
    import subprocess, time, urllib.request, re
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=True)
    log_file = open("streamlit_log.txt", "w")
    streamlit_process = subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(APP_DIR / "app.py"),
                                          "--server.port", "8501", "--server.headless", "true"], stdout=log_file, stderr=log_file)
    time.sleep(10)
    subprocess.run("npm install -g localtunnel", shell=True, capture_output=True)
    tunnel = subprocess.Popen(["npx", "localtunnel", "--port", "8501"], stdout=subprocess.PIPE, text=True)
    url = tunnel.stdout.readline().strip()
    password = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
    print(url)
    print(f"If the page asks for a tunnel password, enter: {password}")
    print("Stop the preview with: streamlit_process.terminate(); tunnel.terminate()")

## 33. GitHub, Streamlit Community Cloud and the Final Package

### 33.1 Put the project on GitHub
You can either **upload manually** (recommended for beginners) or **push from Colab**:

**Manual upload**
1. Download the package created at the end of this notebook, unzip it and upload the files to a new **public** GitHub repository (or add your instructor as a collaborator if it is private).
2. Add this notebook under `notebooks/` (download it with *File ▸ Download ▸ .ipynb*; use a numbered name such as `01_stages_1_to_7.ipynb`).
3. Every team member must **commit their own work with meaningful messages** – the commit history is the evidence of your contribution.

**Push from Colab (optional)** – use the next cell. It needs a GitHub *personal access token* stored in **Colab Secrets** (key icon in the left bar, name `GITHUB_TOKEN`, notebook access ON). **Never type the token into a cell and never commit it.**

### 33.2 Publish on Streamlit Community Cloud
1. Go to <https://share.streamlit.io> and sign in with GitHub.
2. Choose **Create app** (or *New app*) ▸ select your repository and branch (`main`).
3. Set the **main file path** to `app/app.py`. The file `requirements.txt` must be in the repository root.
4. Click **Deploy** and watch the logs. If a library version fails, loosen the pin in `requirements.txt` and push the change.
5. Test the live link with normal and extreme inputs, then paste the link into `README.md` and on the first page of the report.

> Streamlit Community Cloud deploys **directly from your GitHub repository** – this is a rubric requirement, so do not upload the app anywhere else.

In [8]:
# OPTIONAL: push the project folder to GitHub from Colab (disabled by default)
PUSH_TO_GITHUB = False
GITHUB_REPO = "https://github.com/<your-username>/<your-repository>.git"     # edit
GIT_USER_NAME = "<your name>"                                                  # edit
GIT_USER_EMAIL = "<your email>"                                                # edit
COMMIT_MESSAGE = "Add Stage 1-7 notebook outputs, model and Streamlit app"

if PUSH_TO_GITHUB:
    import subprocess
    if "<" in GITHUB_REPO or "<" in GIT_USER_NAME:
        raise ValueError("Edit GITHUB_REPO, GIT_USER_NAME and GIT_USER_EMAIL first.")
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")           # stored in Colab Secrets - never printed

    # Safety check: GitHub rejects files above 100 MB (ignored paths are skipped)
    ignored = [PROJECT_DIR / "data/raw", PROJECT_DIR / "data/processed/vehicles_cleaned.csv"]
    too_big = [p for p in PROJECT_DIR.rglob("*") if p.is_file() and p.stat().st_size > 95 * 1024**2
               and not any(str(p).startswith(str(i)) for i in ignored)]
    if too_big:
        raise RuntimeError(f"These files are too large for GitHub: {[str(p) for p in too_big]}")

    def git(*args):
        result = subprocess.run(["git", *args], cwd=PROJECT_DIR, capture_output=True, text=True)
        print((result.stdout + result.stderr).replace(token, "***").strip())
        return result

    git("init")
    git("config", "user.name", GIT_USER_NAME)
    git("config", "user.email", GIT_USER_EMAIL)
    git("add", ".")
    git("commit", "-m", COMMIT_MESSAGE)
    git("branch", "-M", "main")
    git("remote", "remove", "origin")
    git("remote", "add", "origin", GITHUB_REPO.replace("https://", f"https://{token}@"))
    git("push", "-u", "origin", "main")
else:
    print("GitHub push is disabled (PUSH_TO_GITHUB = False). Use the manual upload described above.")

GitHub push is disabled (PUSH_TO_GITHUB = False). Use the manual upload described above.


In [10]:
# Final package: everything for GitHub except the raw and the large full cleaned CSV
package_dir = Path("final_project_package")
shutil.rmtree(package_dir, ignore_errors=True)
shutil.copytree(PROJECT_DIR, package_dir,
                ignore=shutil.ignore_patterns("raw", "vehicles_cleaned.csv", "__pycache__", ".git"))
zip_path = shutil.make_archive("Used-Vehicle-Price-Prediction", "zip", package_dir)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size / 1024**2:,.1f} MB)\n")
print("Contents:")
for path in sorted(package_dir.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(package_dir)}  ({path.stat().st_size / 1024:,.0f} KB)")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

Created /content/Used-Vehicle-Price-Prediction.zip (4.9 MB)

Contents:
  .gitignore  (0 KB)
  README.md  (2 KB)
  app/app.py  (7 KB)
  data/README_DATA.md  (1 KB)
  data/sample/vehicles_cleaned_sample.csv  (1,050 KB)
  docs/data_dictionary.csv  (6 KB)
  docs/feature_importance.png  (70 KB)
  models/model_metadata.json  (38 KB)
  models/price_model.joblib  (3,017 KB)
  outputs/before_after_quality_report.csv  (2 KB)
  outputs/cleaning_audit_log.csv  (10 KB)
  outputs/cleaning_settings.json  (0 KB)
  outputs/cleaning_summary.csv  (0 KB)
  outputs/data_dictionary.csv  (6 KB)
  outputs/duplicate_report.csv  (1 KB)
  outputs/eda_driver_strength.csv  (1 KB)
  outputs/excluded_price_records_log.csv  (453 KB)
  outputs/feature_engineering_experiment.csv  (0 KB)
  outputs/feature_engineering_table.csv  (1 KB)
  outputs/figures/01_price_distribution.png  (73 KB)
  outputs/figures/02_price_by_year.png  (91 KB)
  outputs/figures/03_price_vs_odometer.png  (213 KB)
  outputs/figures/04_price_by_cate

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 33.3 Report outline (Deliverable 1 – group report, PDF)
Colab cannot write your report, but it has produced the evidence for each section:

| Report section | What to write | Evidence produced by this notebook |
|---|---|---|
| First page | GitHub link and live Streamlit link | `README.md` |
| Stage 1 | Business problem, objectives, expected outcomes | Section 1 |
| Stage 2 | Dataset source, licence, justification, search evidence | Stage 2 section and requirement table |
| Stage 3 | Quality problems, decisions, before/after | `outputs/cleaning_audit_log.csv`, `before_after_quality_report.csv`, `missing_value_report.csv`, `duplicate_report.csv` |
| Stage 4 | Patterns, features, business meaning | `outputs/figures/01…09_*.png`, `feature_engineering_table.csv` |
| Stage 5 | Models compared, justification | `model_comparison.csv`, `feature_engineering_experiment.csv`, figure `10_model_comparison.png` |
| Stage 6 | Tuning, metrics, error analysis | `tuning_results.csv`, `tuning_before_after.csv`, `final_metrics.csv`, `target_check.csv`, figures `11…12` |
| Stage 7 | App description, deployment, screenshots | `app/app.py`, live app |
| Also required | Dataset source and licence, **list of AI tools used and how**, references | Fill in the README and the report |

### 33.4 Individual assessment (Deliverable 2)
Parts A, B and C must be written **independently, in your own words** – the notebook can remind you of the facts (numbers, tables, charts) but not replace your explanations. Be ready to explain every cell orally.

In [11]:
print("NOTEBOOK COMPLETE")
print("-" * 60)
print(f"Stage 3  cleaned records           : {cleaning_settings['records_cleaned']:,}")
print(f"Stage 6  final model               : {FINAL_MODEL_NAME}")
print(f"         test R2 / MdAPE / within20: {final_metrics['R2']:.3f} / {final_metrics['MdAPE (%)']:.1f}% / {final_metrics['Within 20% (%)']:.1f}%")
print(f"Stage 7  project folder            : {PROJECT_DIR}")
print("Next: upload to GitHub, deploy on Streamlit Community Cloud, add the links to README and the report.")

NOTEBOOK COMPLETE
------------------------------------------------------------
Stage 3  cleaned records           : 240,186
Stage 6  final model               : Hist. gradient boosting
         test R2 / MdAPE / within20: 0.676 / 18.7% / 52.7%
Stage 7  project folder            : /content/drive/MyDrive/Used-Vehicle-Price-Prediction
Next: upload to GitHub, deploy on Streamlit Community Cloud, add the links to README and the report.
